# Graded Mini Project (Part B): Retail Transaction Insights

This notebook performs data preparation, exploration, analysis, and visualizations for **Retail_Transactions_Dataset.csv**.

**Deliverables covered:**
- Clean, commented Python analysis
- Required outputs and plots (city bar chart, payment method pie chart, monthly revenue trend, seasonal analysis, etc.)
- Concise summary of key insights and recommended actions

In [ ]:
# =========================
# Part B: Retail Transaction Insights
# =========================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import ast
from collections import Counter

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [ ]:
# Task 1: Data Preparation
# Step 1: Load the CSV file

file_path = "Retail_Transactions_Dataset.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

In [ ]:
# Inspect columns, dtypes, and missing values
print("Columns:", list(df.columns))
df.info()

print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False))

In [ ]:
# Parse and convert Date column into datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Extract additional useful fields
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.month_name()
df["DayOfWeek"] = df["Date"].dt.day_name()
df["YearMonth"] = df["Date"].dt.to_period("M").astype(str)

df[["Date","Year","Month","Month_Name","DayOfWeek","YearMonth"]].head()

In [ ]:
# Clean & preprocess numeric columns
df["Total_Items"] = pd.to_numeric(df["Total_Items"], errors="coerce")
df["Total_Cost"] = pd.to_numeric(df["Total_Cost"], errors="coerce")

# Normalize Discount_Applied values into True/False
def normalize_discount(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["true", "1", "yes", "y"]:
        return True
    if s in ["false", "0", "no", "n"]:
        return False
    return np.nan

df["Discount_Applied"] = df["Discount_Applied"].apply(normalize_discount)

df[["Total_Items", "Total_Cost", "Discount_Applied"]].head()

In [ ]:
# Product column is stored as a list-like string, e.g. "['Ketchup', 'Milk']"
# Parse it safely into a real Python list.

def parse_product_list(x):
    if pd.isna(x):
        return []
    try:
        parsed = ast.literal_eval(str(x))
        if isinstance(parsed, list):
            return [str(p).strip() for p in parsed if str(p).strip()]
        return [str(parsed).strip()]
    except Exception:
        return [str(x).strip()]

df["Product_List"] = df["Product"].apply(parse_product_list)
df[["Product", "Product_List"]].head(8)

In [ ]:
# Drop rows missing critical columns for analysis
critical_cols = ["Date", "Total_Cost", "City", "Payment_Method", "Customer_Name"]
df_clean = df.dropna(subset=critical_cols).copy()

print("Rows before:", len(df))
print("Rows after :", len(df_clean))
df_clean.head()

## Task 2: Basic Exploration

In [ ]:
# Total transactions + unique customers
total_transactions = df_clean["Transaction_ID"].nunique()
unique_customers = df_clean["Customer_Name"].nunique()

print("Total transactions:", total_transactions)
print("Unique customers:", unique_customers)

In [ ]:
# Top 5 most common products across all transactions
all_products = [p for row in df_clean["Product_List"] for p in row]
top5_products = Counter(all_products).most_common(5)

print("Top 5 most common products:")
for prod, cnt in top5_products:
    print(f"- {prod}: {cnt}")

In [ ]:
# Cities with the highest number of transactions
city_counts = df_clean["City"].value_counts()
print("Top cities by transaction count:")
display(city_counts.head(10))

## Task 3: Customer Behaviour Analysis

In [ ]:
# Which customer categories spend the most on average?
avg_spend_by_category = df_clean.groupby("Customer_Category")["Total_Cost"].mean().sort_values(ascending=False)
print("Average spend by customer category:")
display(avg_spend_by_category)

In [ ]:
# Do certain customer categories prefer specific payment methods?
payment_pref_counts = pd.crosstab(df_clean["Customer_Category"], df_clean["Payment_Method"])
payment_pref_pct = payment_pref_counts.div(payment_pref_counts.sum(axis=1), axis=0) * 100

print("Payment method preference (% within each category):")
display(payment_pref_pct.round(2))

In [ ]:
# Average number of items bought per transaction per store type
avg_items_by_store = df_clean.groupby("Store_Type")["Total_Items"].mean().sort_values(ascending=False)
print("Average number of items per transaction by store type:")
display(avg_items_by_store)

## Task 4: Promotion & Discount Impact

In [ ]:
# Average cost where discount applied vs not applied
avg_cost_discount = df_clean.groupby("Discount_Applied")["Total_Cost"].mean()
print("Average transaction cost by Discount_Applied:")
display(avg_cost_discount)

In [ ]:
# Compare average number of items purchased for different promotion types
avg_items_by_promo = df_clean.groupby("Promotion")["Total_Items"].mean().sort_values(ascending=False)
print("Average items purchased by Promotion type:")
display(avg_items_by_promo)

In [ ]:
# Which promotion seems most effective (highest avg Total_Cost)?
avg_cost_by_promo = df_clean.groupby("Promotion")["Total_Cost"].mean().sort_values(ascending=False)
print("Average Total_Cost by Promotion type:")
display(avg_cost_by_promo)

## Task 5: Seasonality Trends

In [ ]:
# Which season has the highest total revenue?
revenue_by_season = df_clean.groupby("Season")["Total_Cost"].sum().sort_values(ascending=False)
print("Total revenue by Season:")
display(revenue_by_season)

In [ ]:
# Seasonal preferences for store types (transaction counts)
season_store_counts = pd.crosstab(df_clean["Season"], df_clean["Store_Type"])
print("Transaction counts by Season and Store_Type:")
display(season_store_counts)

In [ ]:
# Plot: average spending per season
avg_spend_season = df_clean.groupby("Season")["Total_Cost"].mean().sort_values(ascending=False)

plt.figure(figsize=(7,4))
sns.barplot(x=avg_spend_season.index, y=avg_spend_season.values, palette="viridis")
plt.title("Average Spending per Season")
plt.xlabel("Season")
plt.ylabel("Average Total Cost")
plt.show()

print("Observation: Higher-spending seasons are good targets for premium bundles and stronger campaigns.")

## Task 6: Visualisation Dashboard (Required Plots)

In [ ]:
# Bar chart of number of transactions per city (Top 10)
top_cities = df_clean["City"].value_counts().head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=top_cities.index, y=top_cities.values, palette="magma")
plt.title("Top 10 Cities by Number of Transactions")
plt.xlabel("City")
plt.ylabel("Transactions")
plt.xticks(rotation=45, ha="right")
plt.show()

print("Observation: Focus inventory and promotions in high-transaction cities.")

In [ ]:
# Pie chart: distribution of payment methods
payment_counts = df_clean["Payment_Method"].value_counts()

plt.figure(figsize=(7,7))
plt.pie(payment_counts.values, labels=payment_counts.index, autopct="%1.1f%%", startangle=140)
plt.title("Payment Method Distribution")
plt.show()

print("Observation: Payment method preference helps optimize checkout experience and offers.")

In [ ]:
# Line chart: monthly revenue trend
monthly_revenue = df_clean.groupby("YearMonth")["Total_Cost"].sum().sort_index()

plt.figure(figsize=(12,5))
monthly_revenue.plot(marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Year-Month")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45, ha="right")
plt.show()

print("Observation: Peaks/dips may be linked to seasons or promotions.")

In [ ]:
# Heatmap: revenue by season and customer category
pivot_rev = df_clean.pivot_table(
    index="Season",
    columns="Customer_Category",
    values="Total_Cost",
    aggfunc="sum",
    fill_value=0
)

plt.figure(figsize=(10,5))
sns.heatmap(pivot_rev, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Revenue by Season and Customer Category")
plt.xlabel("Customer Category")
plt.ylabel("Season")
plt.show()

print("Observation: High-revenue combinations can be targeted with tailored campaigns.")

## Concise Summary: Key Insights & Recommended Actions

In [ ]:
top_city = df_clean["City"].value_counts().idxmax()
top_payment = df_clean["Payment_Method"].value_counts().idxmax()
top_category_avg = df_clean.groupby("Customer_Category")["Total_Cost"].mean().idxmax()
top_season_rev = df_clean.groupby("Season")["Total_Cost"].sum().idxmax()

print("=== Key Insights & Recommended Actions ===")
print(f"1) Highest transaction city: {top_city}. Action: prioritize stock planning and localized promotions there.")
print(f"2) Most common payment method: {top_payment}. Action: ensure it is always supported and consider payment-linked offers.")
print(f"3) Highest average spending category: {top_category_avg}. Action: target this segment with premium bundles and loyalty rewards.")
print(f"4) Highest revenue season: {top_season_rev}. Action: allocate higher marketing budget and staffing for this season.")